# Phase 1 - behavioral fidelity: greedy vs sampling (no retrain)

Fresh Colab, **A100 + High-RAM strongly recommended** (CELL 2 builds `base_resized` by loading the base in fp32 on CPU ~12GB, and CELL 3 free-runs generation over the holdout twice), then **Run All**.

Reloads the P6 two-stage adapter from HF and runs `sft.score_tokens` with `--behavioral-sampling both`, so you get, on the same unit-aware holdout:
- the teacher-forced `METRIC=` (token accuracy, unchanged locked contract), and
- **behavioral fidelity** (decode -> re-measure behavior_v2 -> direction+magnitude agreement vs the requested spec) under **greedy** vs **sampling t=0.7** + collapse-rate + decoded deltaE.

This is the real test Phase 0 pointed to: does sampling recover the *requested behavior* (not just RMS strength)? Scored with `attribute_spec_text` conditioning (via `configs/candidate_two_stage.json`) to match how P6 was trained.

The only long wait is the one-time ~9.85GB corpus stage. Set `BEHAVIORAL_LIMIT = 0` in CELL 3 for the full holdout.


In [1]:
# CELL 1 - provision the runtime (clone, install, HF token, stage the corpus)
import os, pathlib, subprocess, sys
REPO = '/content/SLM'
BRANCH = 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = _env_token('HF_TOKEN') or getpass.getpass('HF_TOKEN (read, for staging + adapter): ').strip()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')


HEAD: 1bfe1c9 Add Phase 1 behavioral-score notebook (greedy vs sampling, no retrain)
HF_TOKEN set: True
corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...


In [2]:
# CELL 2 - build base_resized (score_tokens needs it) + download the P6 two-stage adapter from HF
import os, pathlib, subprocess, sys, json
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
D = pathlib.Path('models/base_resized')
if not ((D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file()):
    print('building base_resized (loads base in fp32 on CPU ~12GB; A100/High-RAM recommended) ...')
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
vr = json.load(open(D / 'vocab_resize_manifest.json'))
assert vr.get('tokenizer_version') == 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f', vr.get('tokenizer_version')
print('base_resized OK; identity:', vr['tokenizer_version'])

from huggingface_hub import snapshot_download
REPO_ID = 'ericrcwu/LUT_SLM_sft_adapters'
ADAPTER_SUBFOLDER = 'p6_twostage_d0f9c744_smokefull'
snapshot_download(repo_id=REPO_ID, allow_patterns=[ADAPTER_SUBFOLDER + '/*'],
                  local_dir='models/sft_adapters', token=os.environ.get('HF_TOKEN'))
ADAPTER = 'models/sft_adapters/' + ADAPTER_SUBFOLDER
am = json.load(open(pathlib.Path(ADAPTER) / 'adapter_manifest.json'))
assert am['tokenizer_version'] == vr['tokenizer_version'], 'adapter/tokenizer identity mismatch'
assert pathlib.Path(ADAPTER, 'adapter_model.safetensors').is_file(), ADAPTER
print('adapter ready:', ADAPTER, '| step:', am.get('adapter_step'), '| sha:', am.get('adapter_sha256', '')[:16])


building base_resized (loads base in fp32 on CPU ~12GB; A100/High-RAM recommended) ...
base_resized OK; identity: vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

adapter ready: models/sft_adapters/p6_twostage_d0f9c744_smokefull | step: 182 | sha: 7a4f8a1a55d8dba4


In [3]:
# CELL 3 - score P6: teacher-forced METRIC + behavioral fidelity under greedy vs sampling t=0.7
import os, sys, json, subprocess
ADAPTER = 'models/sft_adapters/p6_twostage_d0f9c744_smokefull'
BEHAVIORAL_LIMIT = 64   # rows in the (slow) free-running pass; 0 = full unit-aware holdout

env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'sft.score_tokens',
       '--config', 'configs/candidate_two_stage.json',   # input_field=attribute_spec_text (matches P6)
       '--resized-model', 'models/base_resized',
       '--adapter', ADAPTER,
       '--limit', '0',                                    # full holdout for the teacher-forced METRIC
       '--behavioral-sampling', 'both',
       '--behavioral-limit', str(BEHAVIORAL_LIMIT),
       '--behavioral-temperature', '0.7', '--behavioral-top-p', '0.9']
print('running:', ' '.join(cmd), '\n')
p = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(p.stdout[-9000:])
if p.returncode != 0:
    print('--- STDERR tail ---\n', p.stderr[-3000:])

summary = None
for line in p.stdout.splitlines():
    if line.strip().startswith('{') and 'score_summary' in line:
        try:
            summary = json.loads(line)['score_summary']
        except Exception:
            pass
if summary:
    print('\n================= SUMMARY =================')
    print('teacher-forced token_accuracy (METRIC):', round(summary.get('metric', 0.0), 4),
          '(one-stage baseline 0.362)')
    for key, tag in (('behavioral', 'greedy'), ('behavioral_sampled', 'sample t=0.7')):
        b = summary.get(key) or {}
        if 'behavioral_fidelity_mean' in b:
            print('  %-13s fidelity=%s  collapse_rate=%s  resid_median=%s  deltaE=%s  scored=%s refused=%s' % (
                tag, b.get('behavioral_fidelity_mean'), b.get('collapse_rate'),
                b.get('residual_norm_median'), b.get('decoded_delta_e_mean'),
                b.get('scored'), b.get('refused')))
    print('\nRead: higher behavioral fidelity under sample-t0.7 than greedy => decoding regime is the')
    print('      lever (do the training fixes). Both low => input/loss are the bottleneck.')


running: /usr/bin/python3 -m sft.score_tokens --config configs/candidate_two_stage.json --resized-model models/base_resized --adapter models/sft_adapters/p6_twostage_d0f9c744_smokefull --limit 0 --behavioral-sampling both --behavioral-limit 64 --behavioral-temperature 0.7 --behavioral-top-p 0.9 

[score][behavioral:greedy] fidelity=0.159301046686157 collapse_rate=0.9375 resid_median=0.06915280861492439 scored=64 refused=0
[score][behavioral:t=0.7] fidelity=0.09108920720823294 collapse_rate=0.140625 resid_median=0.09422162395259359 scored=64 refused=0
{"score_summary": {"metric": 0.32864583333333336, "token_accuracy": 0.32864583333333336, "overall_ci_low": 0.30120446044271365, "overall_ci_high": 0.354812615972024, "macro_family_accuracy": 0.29036096643518516, "exact_match_rate": 0.0, "code_positions": 7680, "correct": 2524, "scored_rows": 120, "scored_units": 97, "per_family": {"fivek_derived": {"accuracy": 0.3564453125, "ci_low": 0.32083333333333336, "ci_high": 0.3865182976973684, "row

In [5]:
# Teacher-forced-argmax behavioral fidelity — exposure-bias isolation (cheap, no retrain).
# Decodes the model's one-step-ahead argmax codes (perfect gold prefix every step) and scores
# their behavior vs the requested spec. HIGH here + low free-running => exposure bias is THE problem
# (fix = scheduled sampling / free-running FT). LOW here too => a perfect prefix still can't render
# the spec => input/loss/capacity is the bottleneck.
import os, numpy as np, torch
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
os.chdir('/content/SLM')

# load model+adapter if this is a fresh session (in-memory resize; T4-safe). Reuses gen/proc if present.
if 'gen' not in globals() or 'proc' not in globals():
    from transformers import (AutoProcessor, AutoTokenizer, BitsAndBytesConfig,
                              Qwen2_5_VLForConditionalGeneration)
    from peft import PeftModel
    BASE = 'Qwen/Qwen2.5-VL-3B-Instruct'
    ADAPTER = 'models/sft_adapters/p6_twostage_d0f9c744_smokefull'
    MINP, MAXP, VOCAB = 3136, 200704, 151924
    tkz = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
    canon = ['<lut_bos>', '<lut_eos>', '<unsupported>'] + ['<lut_%03d>' % i for i in range(256)]
    tkz.add_special_tokens({'additional_special_tokens': canon}); assert len(tkz) == VOCAB
    proc = AutoProcessor.from_pretrained(BASE, trust_remote_code=True, min_pixels=MINP, max_pixels=MAXP)
    proc.tokenizer = tkz
    maj, _ = torch.cuda.get_device_capability(0)
    dt = torch.bfloat16 if maj >= 8 else torch.float16
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dt)
    b = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        BASE, quantization_config=bnb, torch_dtype=dt, device_map={'': 0},
        low_cpu_mem_usage=True, trust_remote_code=True)
    if b.get_input_embeddings().num_embeddings != len(tkz):
        b.resize_token_embeddings(len(tkz), mean_resizing=False)
    gen = PeftModel.from_pretrained(b, ADAPTER, is_trainable=False,
                                    autocast_adapter_dtype=False, low_cpu_mem_usage=True).eval()
    print('loaded model for teacher-forced measurement')

from sft.config import SFTConfig
from sft.example import build_supervised_example, load_rows, supported_rows
from eval.vocab import code_token
from eval.behavioral_fidelity import score_generation, summarize_fidelity
from data_pipeline.attribute_spec import ground_truth_attribute_spec_text

cfg = SFTConfig(input_field='attribute_spec_text')     # match how P6 was trained
LIMIT = 64                                             # same slice as the free-running behavioral run
code_ids = torch.tensor([proc.tokenizer.convert_tokens_to_ids(code_token(k)) for k in range(256)],
                        device=gen.device)

rows = supported_rows(load_rows(cfg.active_rows_path), holdout=True)[:LIMIT]
recs = []
for row in rows:
    try:
        batch = build_supervised_example(proc, row, cfg, device=gen.device,
                                         input_field='attribute_spec_text')
    except Exception as e:
        print('skip', row.get('id'), type(e).__name__); continue
    labels = batch.pop('labels')
    with torch.no_grad():
        logits = gen(**batch).logits
    gold = batch['input_ids'][:, 1:]
    sel = (labels[:, 1:] != -100) & torch.isin(gold, code_ids)
    sel_pos = sel[0].nonzero(as_tuple=True)[0]
    if sel_pos.numel() != 64:
        print('skip non-64', row.get('id'), int(sel_pos.numel())); continue
    tf_logits = logits[0, :-1, :][sel_pos][:, code_ids]      # [64,256] restricted to code vocab
    tf_codes = tf_logits.argmax(-1).tolist()                 # best code per position (perfect prefix)
    spec = ground_truth_attribute_spec_text(row)
    recs.append(score_generation(tf_codes, spec, target_codes=row.get('target_tokens')))

s = summarize_fidelity(recs)
print('\n===== teacher-forced-argmax (exposure-bias ceiling for THIS model) =====')
print('behavioral fidelity : %.3f' % (s['behavioral_fidelity_mean'] or -1))
print('collapse_rate       : %.3f' % (s['collapse_rate'] or -1))
print('resid_median        : %.4f' % (s['residual_norm_median'] or -1))
print('decoded deltaE       : %.2f' % (s['decoded_delta_e_mean'] or -1))
print('scored              :', len(recs))
print('\nfree-running was: greedy 0.159 | sample-t0.7 0.091 | real-code ceiling ~0.89')
print('HIGH here (~0.6-0.8) => exposure bias IS the problem => scheduled sampling / free-running FT.')
print('LOW here (~0.2)      => perfect prefix still cannot render the spec => input/loss/capacity.')


===== teacher-forced-argmax (exposure-bias ceiling for THIS model) =====
behavioral fidelity : 0.708
collapse_rate       : -1.000
resid_median        : 0.0444
decoded deltaE       : 1.13
scored              : 64

free-running was: greedy 0.159 | sample-t0.7 0.091 | real-code ceiling ~0.89
HIGH here (~0.6-0.8) => exposure bias IS the problem => scheduled sampling / free-running FT.
LOW here (~0.2)      => perfect prefix still cannot render the spec => input/loss/capacity.
